# Lesson 16: PCA

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

RANDOM_SEED = 315

## 1. Iris dataset

In [ ]:
dataset_path = 'https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/refs/heads/main/data/clean_iris.csv'
data_df = pd.read_csv(dataset_path)
data_df.head()

### 1.1. 2D feature space

In [ ]:
# Add the species name for the plot
data_df['species'] = data_df['specie'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
data_df.drop('specie', axis=1, inplace=True)

sns.scatterplot(data_df, x='petal width (cm)', y='sepal width (cm)', hue='species')
plt.show()

### 1.2. 3D feature space

In [ ]:
fig = px.scatter_3d(
    data_df,
    x='petal width (cm)',
    y='petal length (cm)',
    z='sepal width (cm)',
    color='species',
    width=700,
    height=700,
    opacity=0.8
)

fig.update_layout()
fig.show()

## 2. Dimensionality reduction: PCA

In [ ]:
N = len(data_df)
x_data = data_df['petal width (cm)']
y_data = data_df['sepal width (cm)']
x_data = np.reshape(x_data, (len(data_df), 1))
y_data = np.reshape(y_data, (len(data_df), 1))
data = np.hstack((x_data, y_data))

mu = data.mean(axis=0)
data = data - mu

eigenvectors, eigenvalues, V = np.linalg.svd(data.T, full_matrices=False)
projected_data = np.dot(data, eigenvectors)
sigma = projected_data.std(axis=0).mean()
axis = eigenvectors[:, 0]
start, end = mu, mu + sigma * axis

pca_df = data_df[['petal width (cm)', 'sepal width (cm)']]
pca = PCA()
pca_data = pca.fit_transform(pca_df)
pca_df = pd.DataFrame(pca_data, columns=['PC1', 'PC2'])

fig, ax = plt.subplots(1,2, figsize=(8, 4))

ax[0].set_title('First principal component: petal & sepal widths')
ax[0].scatter(x_data, y_data, color='grey', label='Data')
ax[0].axline(xy1=start, xy2=end, color='red', linestyle='--', label='Component 1 axis')
ax[0].set_xlabel('Petal Width (cm)')
ax[0].set_ylabel('Sepal Width (cm)')
ax[0].legend()

ax[1].set_title('Distribution of first principal component')
ax[1].hist(pca_df['PC1'], bins=30, color='black')
ax[1].set_xlabel('Component 1 value')
ax[1].set_ylabel('Count')

fig.tight_layout()
plt.show()